# 🧩 Modul 06: Unsupervised Learning I - Klasterisasi (*Clustering*) & Reduksi Dimensi (PCA)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/06_Clustering_and_PCA.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & analogi dunia nyata sebelum rumus matematika dan kode.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── [5. Evaluation] ◄── 🎯 [4. MODELING: CLUSTERING & PCA]
```
Di modul sebelumnya, kita memiliki kunci jawaban ($y$). Namun di dunia nyata, **90% data tidak memiliki label**! Di **Modul 06**, kita membiarkan mesin menjadi detektif mandiri untuk menemukan kelompok alami (*Customer Segments*) dan meringkas dimensi data yang rumit!


## 🕵️ 1. Filosofi Unsupervised Learning: Menemukan Pola Tanpa Kunci Jawaban

### 📚 Analogi Perpustakaan Kuno Tanpa Katalog
Bayangkan Anda mewarisi gudang berisi 10.000 naskah kuno tanpa judul, tanpa penulis, dan tanpa label kategori:
* Dalam **Supervised Learning**, ada guru yang memberi tahu: *"Ini naskah kedokteran, ini naskah astronomi, ini sastra."*
* Dalam **Unsupervised Learning**, Anda **tidak punya guru dan tidak punya kunci jawaban**! Anda harus mengelompokkan naskah-naskah tersebut sendiri berdasarkan kemiripan gaya bahasa, ketebalan kertas, dan topik kata yang sering muncul.

### 💼 Mengapa Klasterisasi (*Clustering*) Begitu Berharga di Industri?
1. **Segmentasi Pelanggan E-Commerce**: Membagi jutaan pembeli menjadi kelompok:
   * *Klaster 1*: Pemburu Promo & Diskon Akhir Bulan.
   * *Klaster 2*: Pembeli Sultan (Sering belanja barang mewah tanpa diskon).
   * *Klaster 3*: Pelanggan Loyal (Rutin beli kebutuhan pokok bulanan).
2. **Sistem Rekomendasi Musik (Spotify)**: Mengelompokkan jutaan lagu berdasarkan kemiripan tempo, instrumen, dan mood.
3. **Deteksi Anomali Siber**: Mengenali pola lalu lintas internet yang aneh karena terpisah jauh dari gerombolan aktivitas normal.


## 🍷 2. Memuat Dataset Nyata: 13 Fitur Kimiawi Anggur (Wine Dataset)

Kita akan menggunakan **Wine Dataset** resmi dari Scikit-Learn. Dataset ini memuat **13 fitur kimiawi kontinu** dari sampel anggur (seperti kadar alkohol, asam malat, abu mineral, magnesium, flavonoid, intensitas warna, hingga kadar proline).

> ⚠️ **Hukum Wajib Klasterisasi**:  
> Seluruh algoritma klasterisasi berbasis jarak geometris (Euclidean) **WAJIB menggunakan Feature Scaling (StandardScaler)**!  
> Jika kadar magnesium bernilai 100 dan kadar asam malat bernilai 1.5, magnesium akan mendominasi perhitungan jarak dan membutakan variabel lainnya.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

# Memuat dataset anggur
wine_raw = load_wine(as_frame=True)
df_features = wine_raw.data # Kita HANYA mengambil fitur (X), mengabaikan kolom label target asli!

print(f"Dimensi Data: {df_features.shape[0]} sampel anggur × {df_features.shape[1]} fitur kimiawi")
print("\n5 Baris Sampel Fitur:")
df_features.head()


In [ ]:
# Standarisasi Seluruh Fitur (StandardScaler: Rata-rata = 0, Deviasi Standar = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

print("✅ Standarisasi fitur berhasil dilakukan.")
print(f"Bentuk Matriks Berskala: {X_scaled.shape}")
print(f"Rata-rata fitur pertama setelah scale: {X_scaled[:, 0].mean():.4f} (Mendekati 0)")
print(f"Standar deviasi fitur pertama setelah scale: {X_scaled[:, 0].std():.4f} (Mendekati 1)")


## 🎯 3. K-Means Clustering: Membuka Gudang Logistik Baru

### 🚚 Analogi Dunia Nyata: Membuka $K$ Cabang Gudang
Bayangkan sebuah perusahaan logistik ingin membuka **$K=3$ cabang gudang baru**:
1. Perusahaan menaruh 3 titik acak (*Centroid*) di peta.
2. Setiap rumah pelanggan ditugaskan ke gudang terdekat.
3. Perusahaan menghitung ulang titik tengah dari seluruh pelanggan di masing-masing wilayah, lalu menggeser lokasi gudang ke titik tengah tersebut (*Update Centroids*).
4. Langkah 2 dan 3 diulang terus menerus hingga lokasi gudang tidak bergeser lagi (*Konvergensi*).

---

### 📉 Menentukan Jumlah $K$ Optimal: Dual Check (Elbow & Silhouette)
1. **Metode Siku (*Elbow Method*)**: Menghitung **Inersia** (WCSS - *Within-Cluster Sum of Squares*), yaitu total kuadrat jarak sampel ke titik pusatnya. Kita mencari titik "tekukan siku" di mana penurunan inersia mulai melambat drastis.
2. **Skor Silhouette (*Silhouette Score*)**: Mengukur seberapa rapat titik ke klaster sendiri ($a$) dibandingkan ke klaster tetangga ($b$). Nilai berkisar antara $-1.0$ sampai $+1.0$ (Mendekati $+1.0$ = klaster padat dan terpisah tegas).


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 9)
inertia_list = []
silhouette_list = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_list.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_scaled, km.labels_))

# Visualisasi Dual Plot Penentuan K Optimal
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Plot Elbow Method (Inersia)
ax1.plot(k_range, inertia_list, marker='o', color='royalblue', lw=2)
ax1.set_title("1. Metode Siku (Elbow Method)", fontweight='bold')
ax1.set_xlabel("Jumlah Klaster (K)")
ax1.set_ylabel("Inersia (WCSS)")
ax1.axvline(3, color='crimson', linestyle='--', label='Tekukan Siku Optimal (K=3)')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# Plot Silhouette Score
ax2.plot(k_range, silhouette_list, marker='s', color='seagreen', lw=2)
ax2.set_title("2. Analisis Skor Silhouette", fontweight='bold')
ax2.set_xlabel("Jumlah Klaster (K)")
ax2.set_ylabel("Skor Rata-Rata Silhouette")
ax2.axvline(3, color='crimson', linestyle='--', label='Puncak Silhouette (K=3)')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

print(f"🏆 Rekomendasi Matematis: K = 3 adalah jumlah klaster paling optimal (Silhouette Score: {silhouette_list[1]:.3f})!")


## 🌳 4. Hierarchical Clustering (Dendrogram)

### 👨‍👩‍👧‍👦 Analogi Pohon Silsilah Keluarga
Jika K-Means langsung membagi data ke dalam $K$ kelompok sekaligus, **Hierarchical Clustering (Agglomerative)** bekerja bertahap dari bawah ke atas (*Bottom-Up*):
1. Setiap sampel anggur awalnya dianggap sebagai **1 klaster independen**.
2. Dua sampel yang paling mirip digabungkan menjadi satu pasangan klaster.
3. Pasangan-pasangan tersebut terus digabungkan langkah demi langkah hingga akhirnya seluruh sampel bermuara pada satu batang pohon raksasa!

Bagan pohon ini disebut **Dendrogram**. Kita dapat memotong garis horizontal pada ketinggian tertentu untuk menentukan berapa banyak klaster yang terbentuk.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# Menghitung matriks jarak penggabungan menggunakan metode Ward (meminimalkan variansi)
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(
    linked,
    orientation='top',
    distance_sort='descending',
    show_leaf_counts=True,
    no_labels=True,
    color_threshold=10 # Batas potong warna klaster
)
plt.title("Visualisasi Pohon Silsilah Klaster (Dendrogram)", fontsize=13, fontweight='bold')
plt.xlabel("Sampel Anggur (178 Sampel)")
plt.ylabel("Jarak Euklides Penggabungan")
plt.axhline(y=10, color='red', linestyle='--', label='Garis Potong Klaster (Menghasilkan 3 Kelompok Utama)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()

print("Interpretasi Dendrogram:")
print("👉 Memotong dendrogram pada jarak y=10 membagi data menjadi 3 cabang warna utama (Hijau, Merah, Biru), konsisten dengan temuan K-Means!")


## 🌧️ 5. DBSCAN: Klaster Berbasis Kepadatan & Deteksi Outlier

### 🏟️ Analogi Kerumunan Penonton Konser
K-Means memiliki satu kelemahan fatal: **K-Means memaksa semua titik harus masuk ke salah satu klaster**, meskipun titik tersebut sebenarnya adalah data pencilan / outlier ekstrem!

Algoritma **DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*)** bekerja dengan logika kerumunan:
* **Core Point**: Titik yang memiliki minimal `min_samples` tetangga di dalam radius lingkaran `eps`.
* **Border Point**: Titik di pinggiran kerumunan.
* **Noise / Outlier (Label = -1)**: Titik yang menyendiri jauh di sudut tribun tanpa tetangga dekat. Titik ini **tidak dipaksa masuk klaster**, melainkan langsung ditandai sebagai anomali!


In [ ]:
from sklearn.cluster import DBSCAN

# Inisialisasi DBSCAN (eps: jarak radius tetangga, min_samples: batas kepadatan minimum)
dbscan = DBSCAN(eps=2.3, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = list(labels_db).count(-1)

print("=== HASIL KLASTERISASI DBSCAN ===")
print(f"Jumlah Klaster Ditemukan : {n_clusters_db} klaster padat")
print(f"Jumlah Titik Anomali/Noise : {n_noise_db} sampel (diberi label -1)")


## 📸 6. Reduksi Dimensi dengan PCA (*Principal Component Analysis*)

### 🗿 Analogi Memotret Patung 3D ke Foto 2D
Dataset Anggur kita memiliki **13 fitur kimiawi (13 dimensi)**. Mata manusia hanya bisa melihat grafik 2D atau 3D! Bagaimana cara kita memvisualisasikan sebaran klaster 13 dimensi di layar monitor?

Jawabannya adalah **PCA (Principal Component Analysis)**:
* Bayangkan sebuah patung 3 dimensi. Anda mencari sudut bidikan kamera paling strategis, sehingga bayangan patung di dinding 2D tetap memperlihatkan lekukan tubuh dan detail utama patung dengan jelas tanpa kehilangan esensi bentuknya.
* PCA memutar sumbu data dan membuat sumbu komposit baru (**Principal Component 1 & 2**) yang menangkap **persentase variansi data terbesar**!


In [ ]:
from sklearn.decomposition import PCA

# Latih K-Means final dengan K=3
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# Proyeksikan 13 dimensi data ke dalam 2 Komponen Utama (2D)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Proyeksikan juga titik pusat klaster (Centroid) ke ruang 2D PCA
centroids_pca = pca.transform(kmeans_final.cluster_centers_)

var_rasio = pca.explained_variance_ratio_
print("=== REDUKSI DIMENSI PCA (13D ➔ 2D) ===")
print(f"Variansi PC1: {var_rasio[0]*100:.2f}%")
print(f"Variansi PC2: {var_rasio[1]*100:.2f}%")
print(f"Total Variansi Asli yang Berhasil Dipertahankan: {sum(var_rasio)*100:.2f}%!")

# Visualisasi Scatter Plot 2D dari Hasil Klasterisasi
plt.figure(figsize=(9, 6))
warna_klaster = ['#e74c3c', '#3498db', '#2ecc71']
label_klaster = ['Klaster 0', 'Klaster 1', 'Klaster 2']

for i in range(3):
    plt.scatter(
        X_pca[cluster_labels == i, 0],
        X_pca[cluster_labels == i, 1],
        s=60,
        c=warna_klaster[i],
        label=label_klaster[i],
        alpha=0.7,
        edgecolor='k'
    )

# Plot Centroid sebagai bintang kuning besar
plt.scatter(
    centroids_pca[:, 0],
    centroids_pca[:, 1],
    s=250,
    c='gold',
    marker='*',
    edgecolor='black',
    linewidth=1.5,
    label='Centroid (Titik Pusat)',
    zorder=10
)

plt.title("Visualisasi 2D Hasil Klasterisasi K-Means Menggunakan Proyeksi PCA", fontsize=12, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({var_rasio[0]*100:.1f}% Variansi)")
plt.ylabel(f"Principal Component 2 ({var_rasio[1]*100:.1f}% Variansi)")
plt.legend(frameon=True)
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()


## 💼 7. Profiling Bisnis: Membaca Karakteristik Unik Setiap Klaster

Dalam industri data science, pekerjaan Anda **belum selesai** saat grafik klaster terbentuk. Pihak manajemen bisnis akan bertanya:  
*"Apa arti praktis dari Klaster 0, Klaster 1, dan Klaster 2 ini? Apa tindak lanjut bisnisnya?"*

Mari hitung rata-rata fitur kimiawi pada setiap klaster untuk menyusun profil bisnisnya!


In [ ]:
# Gabungkan label klaster kembali ke DataFrame asli
df_profil = df_features.copy()
df_profil['Klaster'] = cluster_labels

# Ambil 5 fitur paling krusial untuk profiling
fitur_kunci = ['alcohol', 'malic_acid', 'flavanoids', 'color_intensity', 'proline']
profil_mean = df_profil.groupby('Klaster')[fitur_kunci].mean()

print("=== RATA-RATA FITUR KIMIAWI PER KLASTER ===")
print(profil_mean.round(2).to_string())

print("\n📋 NAMA DAN REKOMENDASI SEGMEN BISNIS:")
print("🍇 Klaster 0: 'Anggur Ringan Ekonomis' -> Kadar alkohol dan proline terendah, warna lebih cerah.")
print("🍷 Klaster 1: 'Anggur Fermentasi Tajam' -> Asam malat tertinggi, flavonoid rendah (rasa tajam/sepat).")
print("🍾 Klaster 2: 'Anggur Sultan Premium'  -> Kadar alkohol (13.7%) dan proline tertinggi, warna sangat pekat.")


## 🧠 8. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep inti Unsupervised Learning, lalu jalankan sel kode di bawah untuk memeriksa kunci jawabannya:

---
* **Soal 1**: Mengapa pada K-Means dan DBSCAN kita WAJIB melakukan standarisasi fitur (*feature scaling*) sebelum melatih model?
* **Soal 2**: Apa kelemahan utama metode Elbow Method yang berhasil diatasi oleh analisis Silhouette Score?
* **Soal 3**: Mengapa algoritma DBSCAN sangat unggul dibandingkan K-Means pada data yang memiliki banyak pencilan liar (*noise*)?
* **Soal 4**: Apa arti nilai *Explained Variance Ratio* sebesar 55% pada proyeksi 2D PCA?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 6
kunci_jawaban_modul_6 = {
    "Soal 1": "Karena K-Means dan DBSCAN mengukur kemiripan berbasis jarak Euklides. Tanpa scaling, fitur berangka besar (misal gaji atau kadar proline) akan mendominasi perhitungan jarak dan menenggelamkan fitur lainnya.",
    "Soal 2": "Elbow Method sering kali menghasilkan siku yang landai/kabur sehingga sulit menentukan titik belok secara objektif. Silhouette Score memberikan metrik numerik pasti (-1 hingga +1) terkait kepadatan dan pemisahan klaster.",
    "Soal 3": "K-Means memaksa seluruh sampel masuk ke dalam salah satu centroid sehingga outlier merusak titik pusat klaster. DBSCAN mendeteksi klaster berbasis kepadatan dan secara eksplisit melabeli outlier sebagai noise (-1).",
    "Soal 4": "Artinya dari 100% informasi/variabilitas yang tersebar di belasan dimensi asli, 55% di antaranya berhasil dirangkum dan dipertahankan hanya oleh 2 sumbu koordinat baru tersebut."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 06 ===")
for soal, pembahasan in kunci_jawaban_modul_6.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 9. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Apa yang Terjadi Jika K-Means Dilatih Tanpa Standarisasi (*Unscaled*)?
Banyak pemula lupa melakukan `StandardScaler()` sebelum K-Means. Mari buktikan secara empiris mengapa ini adalah kesalahan fatal!

Tugas Anda:
1. Latih model K-Means pada data mentah `df_features` (tanpa standarisasi).
2. Hitung Silhouette Score-nya dan bandingkan dengan data yang sudah di-standarisasi (`X_scaled`).


In [ ]:
# Solusi Eksperimen: Membandingkan K-Means Tanpa Scaling vs Dengan Scaling
km_mentah = KMeans(n_clusters=3, random_state=42, n_init=10)
km_mentah.fit(df_features)
skor_mentah = silhouette_score(df_features, km_mentah.labels_)

km_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)
km_scaled.fit(X_scaled)
skor_scaled = silhouette_score(X_scaled, km_scaled.labels_)

print("=== HASIL EKSPERIMEN STANDARISASI FITUR ===")
print(f"Silhouette Score Data Mentah (Tanpa Scaling) : {skor_mentah:.4f}")
print(f"Silhouette Score Data Berskala (StandardScaler): {skor_scaled:.4f}")

# Cek fitur apa yang membajak klasterisasi data mentah
fitur_dominan = df_features.var().idxmax()
print(f"\n⚠️ Pada data mentah, fitur '{fitur_dominan}' memiliki variansi sebesar {df_features[fitur_dominan].var():,.0f},")
print("sehingga klaster K-Means mentah sebetulnya HANYA mengelompokkan fitur tersebut dan mengabaikan 12 fitur lainnya!")


## 🎯 10. Checklist Kelulusan Modul 06 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami bahwa Unsupervised Learning bekerja tanpa kolom kunci jawaban / label target ($y$).
- [ ] Saya paham mengapa standarisasi fitur (`StandardScaler`) adalah syarat wajib sebelum clustering.
- [ ] Saya bisa menentukan $K$ optimal pada K-Means menggunakan grafik **Elbow Method** dan **Silhouette Score**.
- [ ] Saya memahami cara membaca **Dendrogram** pada Hierarchical Clustering.
- [ ] Saya mengerti keunggulan **DBSCAN** dalam mendeteksi klaster padat sekaligus melabeli anomali/noise (-1).
- [ ] Saya memahami konsep proyeksi 2D menggunakan **PCA** dan cara membaca *Explained Variance Ratio*.
- [ ] Saya mampu menerjemahkan angka rata-rata klaster menjadi label segmen bisnis yang nyata.

---
🚀 **Luar Biasa! Anda telah menuntaskan Modul 06 dengan gemilang!**  
Di modul berikutnya (**Modul 07**), kita akan menjelajahi **Unsupervised Learning II: Aturan Asosiasi (*Association Rule Mining*)**, di mana kita akan membedah strategi penataan produk supermarket menggunakan algoritma legendaris **Apriori** dan **FP-Growth**!
